In [1]:
module DosJarras where

import Data.List (sortOn)


type Estado = (Int, Int)
type Nodo = Estado
type Costo = Int

capA, capB :: Int
capA = 5
capB = 3

-- FUNCIONES DE TRANSFERENCIA
-- Verter de A a B
transferirAB :: Estado -> Estado
transferirAB (a, b) =
    let espacioB = capB - b
        transferir = min a espacioB
    in (a - transferir, b + transferir)

-- Verter de B a A
transferirBA :: Estado -> Estado
transferirBA (a, b) =
    let espacioA = capA - a
        transferir = min b espacioA
    in (a + transferir, b - transferir)


-- GENERACIÓN DE VECINOS
vecinos :: Estado -> [(Estado, Costo)]
vecinos (a, b) =
    [ ((capA, b), 1)            -- Llenar jarra A
    , ((a, capB), 1)            -- Llenar jarra B
    , ((0, b), 1)               -- Vaciar jarra A
    , ((a, 0), 1)               -- Vaciar jarra B
    , (transferirAB (a, b), 1)  -- Verter A → B
    , (transferirBA (a, b), 1)  -- Verter B → A
    ]
    
-- HEURÍSTICA ADMISIBLE
heuristica :: Estado -> Costo
heuristica (a, b)
    | a >= 4 = 0
    | otherwise = max 1 ((4 - a + capB - 1) `div` capB)


-- ALGORITMO A*
aStar :: (Nodo -> [(Nodo, Costo)]) -> (Nodo -> Costo) -> Nodo -> Nodo -> [(Nodo, Costo)]
aStar sucesores heuristica inicio meta = buscar [(inicio, 0)] []
  where
    buscar [] _ = []
    buscar ((nodo, costo):cola) visitados
      | nodo == meta = [(nodo, costo)]
      | nodo `elem` visitados = buscar cola visitados
      | otherwise =
          let nuevos = [ (v, costo + c) | (v, c) <- sucesores nodo, v `notElem` visitados ]
              ordenados = sortOn (\(v, c) -> c + heuristica v) (cola ++ nuevos)
          in (nodo, costo) : buscar ordenados (nodo : visitados)

